# NairaLLM Final V1 — Sequential GPU Training Track (Free Tesla T4)

This notebook trains the single canonical **NairaLLM V1** model sequentially across all 5 stages:

$$\text{Semantic Foundation} \longrightarrow \text{Domain} \longrightarrow \text{Cognition} \longrightarrow \text{Tools} \longrightarrow \text{Behavior} \longrightarrow \mathbf{FINAL\;NAIRALLM\;V1}$$

- **Architecture**: `NairaTransformer` (1,242,880 tied parameters, `vocab_size=1509`, `d_model=128`, `layers=4`, `heads=4`, `d_ff=512`, `SwiGLU`, `RoPE`)
- **Precision**: `FP16_AMP`
- **Hardware Policy**: Google Colab Free Tesla T4 GPU ($0.00)

## 1. Hardware Verification (Tesla T4 & CUDA)

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "ERROR: CUDA not available. Please switch runtime to GPU (T4)!"
print(f"CUDA Active: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB)")

## 2. Environment Setup & Dependency Installation

In [ ]:
%cd /content
# If using Google Drive for persistent checkpoints:
# from google.colab import drive
# drive.mount('/content/drive')

!pip install -q tokenizers pydantic

# Verify path
import os, sys
repo_path = "/content/naira os"
if os.path.exists(repo_path) and repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    %cd "$repo_path"

## 3. Stage 0 — Pre-Flight Zero-Tolerance Verification

In [ ]:
!python NairaLLM/training/scripts/stage_0_preflight.py

## 4. Stage 2 — Naira Domain Alignment
Inherits from `foundation_checkpoint`.

In [ ]:
!python NairaLLM/training/scripts/train_final_v1.py \
    --stage domain \
    --config NairaLLM/configs/final_nairallm_v1.json \
    --parent-checkpoint NairaLLM/training/checkpoints/foundation/foundation_checkpoint_metadata.json

## 5. Stage 3 — Reasoning & Planning Cognition
Inherits from `domain_checkpoint`.

In [ ]:
!python NairaLLM/training/scripts/train_final_v1.py \
    --stage cognition \
    --config NairaLLM/configs/final_nairallm_v1.json \
    --parent-checkpoint NairaLLM/training/checkpoints/domain/nairallm_v1_domain_checkpoint_metadata.json

## 6. Stage 4 — Real Tool Calling & Verification
Inherits from `cognition_checkpoint`.

In [ ]:
!python NairaLLM/training/scripts/train_final_v1.py \
    --stage tools \
    --config NairaLLM/configs/final_nairallm_v1.json \
    --parent-checkpoint NairaLLM/training/checkpoints/cognition/nairallm_v1_cognition_checkpoint_metadata.json

## 7. Stage 5 — Behavior, Autonomy & Safety Boundaries
Inherits from `tools_checkpoint` and produces the frozen `final_v1` checkpoint.

In [ ]:
!python NairaLLM/training/scripts/train_final_v1.py \
    --stage behavior \
    --config NairaLLM/configs/final_nairallm_v1.json \
    --parent-checkpoint NairaLLM/training/checkpoints/tools/nairallm_v1_tools_checkpoint_metadata.json

## 8. Final 144-Prompt Model-Only Benchmark Evaluation

In [ ]:
!python NairaLLM/evaluation/suites/final_v1_benchmark_suite.py \
    --checkpoint NairaLLM/training/checkpoints/behavior/nairallm_v1_behavior_checkpoint.pt \
    --max-tokens 80